# T07. The machine runs

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tamnd/cpython-internals/blob/main/lessons/t07-the-machine-runs/t07.ipynb)

Six lessons of building. Text became tokens, tokens became a tree, the tree got scopes, the scopes became bytecode, and T06 taught you to read the result. Every one of those lessons ended with something built and nothing running. This is the lesson where it runs.

![the eight stages of running Python, with the last stage highlighted](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/t07-the-machine-runs/diagrams/where-we-are.svg)

The part of CPython that runs [bytecode](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#bytecode) is smaller than most people expect. It is one loop, called the [eval loop](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#eval-loop), and it reads an instruction, does what the instruction says, and reads the next one. There is no scheduler, no plan and no lookahead, and everything Python can do is one of about two hundred handlers inside that loop.

By the end you will have watched a real function run one instruction at a time, seen frames appear and disappear as calls are made, and found out why ninety thousand Python calls are fine on your machine while two thousand calls through `sorted` are not.

No C required, and everything here runs on a normal Python.

## About the source references

Now and then this lesson points at CPython's own source, like this: `Python/ceval.c:1212-1218@v3.15.0rc1#_PyEval_EvalFrameDefault`.

Read it as four parts: the file, the lines, the release those line numbers belong to, and the name of the function they are inside.

Every reference is a link, and every one is checked against the pinned source on each change, so a stale reference fails the build instead of sending you somewhere wrong. The function name on the end is what makes the check work. Line numbers move whenever somebody adds code above them, and a moved line number points at something that looks plausible and is not.

You never have to read any of it. The references are there so you can go deeper when you want to, and so you can check that this lesson is not making things up.

## Setup

Colab does not come with the small package these lessons use, so the next cell installs it. If you are running this from a checkout of the repository it is already installed and the cell does nothing.

In [ ]:
import sys

if sys.version_info < (3, 14):
    print("This lesson needs CPython 3.14 or newer.")
    print(f"This runtime is {sys.version.split()[0]}, and the cells below will not run on it.")
else:
    try:
        import pyxray
    except ImportError:
        %pip install -q "pyxray @ git+https://github.com/tamnd/cpython-internals@main#subdirectory=pyxray"
        import pyxray

## Which Python is this

Instruction names and monitoring events both change between releases. Everything here was checked against the version this cell prints.

In [ ]:
import pyxray

pyxray.show()

## The loop

The whole interpreter fits in one picture.

![the interpreter as a ring: read two bytes, look up the handler, run it, move the pointer on](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/t07-the-machine-runs/diagrams/the-loop.svg)

Read two bytes, which gives an opcode and an argument, exactly the encoding T06 pulled apart. Look up the code for that opcode. Run it, which usually means pushing or popping a few things. Move the instruction pointer past the instruction and past any inline cache slots behind it. Then go back to the top.

The C for that ring is two macros. [Python/ceval_macros.h:198-206@v3.15.0rc1#DISPATCH](https://github.com/python/cpython/blob/v3.15.0rc1/Python/ceval_macros.h#L198-L206) is the whole cycle: read the next opcode and argument, then jump to the handler. Every handler ends by calling it.

The function containing all of this is [Python/ceval.c:1212-1218@v3.15.0rc1#_PyEval_EvalFrameDefault](https://github.com/python/cpython/blob/v3.15.0rc1/Python/ceval.c#L1212-L1218). It is around six thousand lines, and almost all of that is the handlers rather than the loop. The loop itself is the four boxes above.

One thing worth noticing early: nothing in that ring says "call a function". A Python function calling a Python function does not leave the loop, and the section on two stacks is where that starts to matter.

## Three ways to get to the handler

"Look up the code for that opcode" hides a choice. That step is called [dispatch](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#dispatch), and CPython does it three different ways depending on how it was built.

![three dispatch strategies with their costs and when CPython uses each](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/t07-the-machine-runs/diagrams/three-ways-to-jump.svg)

The plain version is a `switch` on the opcode inside a `while` loop, which every C compiler understands. The faster version is a [computed goto](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#computed-goto): an array of label addresses, one per opcode, and `goto *opcode_targets[opcode]`. That gives every instruction its own copy of the jump, which the processor's branch predictor can learn separately, and it is worth real percentage points.

The newest version replaces the jump table with a table of functions, and each handler ends by tail calling the next one. Written as a normal call it would grow the C stack forever, so it relies on the compiler turning a tail call into a jump. All three live in the same header: [Python/ceval_macros.h:128-141@v3.15.0rc1#DISPATCH_GOTO](https://github.com/python/cpython/blob/v3.15.0rc1/Python/ceval_macros.h#L128-L141).

Your own build made this choice when it was compiled, and it left a note.

In [ ]:
import sysconfig

arguments = sysconfig.get_config_var("CONFIG_ARGS") or ""
if "--with-tail-call-interp" in arguments:
    print("this build tail calls between handlers")
else:
    print("this build uses computed gotos or a switch")
print()
print("configured with:")
for argument in arguments.replace("'", "").split():
    if argument.startswith("--with") or argument.startswith("--enable"):
        print("   ", argument)

Same instruction set and the same results either way. This is one of the places where CPython is really two or three programs built from the same source, and the difference is invisible from Python except through the note above.

## What one call needs

The loop needs somewhere to keep things. That somewhere is a [frame](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#frame), and it is one block of memory laid out in a fixed order.

![a frame as three stacked regions: specials, locals, and the value stack](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/t07-the-machine-runs/diagrams/a-frame.svg)

The specials are the fixed size part: which code object is running, the globals, the builtins, the previous frame, and `instr_ptr`, which is where we are in the bytecode. Then one slot per local variable. Then `co_stacksize` slots of working space for the [value stack](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#value-stack), which is the number T06 spent half a lesson computing.

The struct is [Include/internal/pycore_interpframe_structs.h:29-53@v3.15.0rc1#_PyInterpreterFrame](https://github.com/python/cpython/blob/v3.15.0rc1/Include/internal/pycore_interpframe_structs.h#L29-L53). The layout is why the interpreter can be fast about locals: `LOAD_FAST 3` is an offset from the start of the frame, worked out at compile time, so there is no dictionary and no name lookup at all.

Frames are not allocated one at a time. They go on a per thread stack, contiguously, so pushing one is usually a pointer bump: [InternalDocs/frames.md:16-25@v3.15.0rc1#Allocation](https://github.com/python/cpython/blob/v3.15.0rc1/InternalDocs/frames.md#L16-L25).

## Calling Python from Python

This is the part that surprises people. When your Python function calls another Python function, the interpreter does not call itself.

![CALL pushes a frame, jumps to the callee, runs the same loop, and RETURN_VALUE pops it](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/t07-the-machine-runs/diagrams/a-call-in-four-moves.svg)

`CALL` pushes a new frame onto that per thread stack and points `instr_ptr` at the callee's first instruction. Then the loop goes round again. It has no idea anything special happened. When the callee hits `RETURN_VALUE`, the frame is popped and `instr_ptr` goes back to where the caller left off.

CPython's own notes describe it in exactly those terms: [InternalDocs/interpreter.md:209-214@v3.15.0rc1#CALL](https://github.com/python/cpython/blob/v3.15.0rc1/InternalDocs/interpreter.md#L209-L214).

This is why `RETURN_VALUE` had a stack effect of zero back in T06. The value is not removed from this frame's stack, because this frame is about to stop existing. It ends up on the caller's stack instead.

## Two stacks, and only one of them is small

There are two stacks in play and they are easy to confuse, so the picture below puts them side by side.

![Python calling Python grows only the data stack, while calling through C grows both](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/t07-the-machine-runs/diagrams/two-stacks.svg)

The data stack is the per thread thing frames live on. It grows as needed and it is cheap. The C stack is the one your operating system handed the thread when it started, usually eight megabytes, and it does not grow.

Python calling Python only touches the first one. Python calling something written in C that calls back into Python touches both, because the C function has a real C stack frame that has to stay put while the callback runs. `sorted` with a `key` is the classic example.

The next cell shows the difference. It takes a few seconds and prints a `RecursionError`, which is the point.

In [ ]:
import sys

sys.setrecursionlimit(200_000)


def only_python(n):
    if n == 0:
        return 0
    return only_python(n - 1)


def through_c(n):
    if n == 0:
        return 0
    return sorted([1], key=lambda _ignored: through_c(n - 1))[0]


print("pure Python, 90000 deep:", only_python(90_000), "no complaints")

depth = 0
try:
    while True:
        depth += 500
        through_c(depth)
except RecursionError as problem:
    print(f"through sorted, gave up somewhere under {depth} deep")
    print("   ", problem)

The message says "Stack overflow" and gives a size in kilobytes, which is CPython telling you it ran out of C stack rather than out of its own recursion budget. The limit you set with `setrecursionlimit` was never reached.

CPython checks this by comparing the current stack pointer against a limit worked out when the thread started, rather than by counting calls: [InternalDocs/stack_protection.md:33-38@v3.15.0rc1#_Py_EnterRecursiveCall](https://github.com/python/cpython/blob/v3.15.0rc1/InternalDocs/stack_protection.md#L33-L38). Counting calls does not work, because different C functions use wildly different amounts of stack.

The number you got is specific to your machine, your build, and how much stack was already in use when the cell started. That is why it is a `RecursionError` and not a promise.

## Watching it happen

Everything so far has been description, and the rest of the lesson is observation.

Since 3.12 there is a supported way to ask the interpreter to tell you what it is doing, and it is a lot better than the old one. You claim a tool id, register callbacks for the events you care about, and turn those events on for a specific code object.

![nine of the twenty sys.monitoring events, with when each fires](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/t07-the-machine-runs/diagrams/what-you-can-watch.svg)

The full list is on `sys.monitoring.events`. Two of the twenty names are not really events: `NO_EVENTS` is zero, and `BRANCH` is the old single event that got split into `BRANCH_LEFT` and `BRANCH_RIGHT`.

Tool ids 0, 1, 2 and 5 are reserved for debuggers, coverage, profilers and the optimizer. 3 and 4 belong to nobody, which is where anything you write should live.

In [ ]:
import sys

monitoring = sys.monitoring

for name in ["DEBUGGER_ID", "COVERAGE_ID", "PROFILER_ID", "OPTIMIZER_ID"]:
    print(f"{name:<14} {getattr(monitoring, name)}")
print()
print("free for you:  3, 4")
print()
events = sorted(name for name in dir(monitoring.events) if name.isupper())
print(len(events), "names on sys.monitoring.events:")
print("   ", ", ".join(events))

## Frames appearing and disappearing

Three events are enough to draw a call tree: a function started, a function returned, a function left because of an exception. Nothing here parses or guesses, and every line is the interpreter reporting a frame being pushed or popped.

In [ ]:
import sys

monitoring = sys.monitoring
event = monitoring.events


def leaf(n):
    if n == 0:
        raise ValueError("bottom")
    return leaf(n - 1)


def top():
    try:
        return leaf(2)
    except ValueError:
        return "caught"


ours = {top.__code__, leaf.__code__}
depth = 0


def started(code, offset):
    global depth
    if code not in ours:
        return
    print("  " * depth + "-> " + code.co_name)
    depth += 1


def returned(code, offset, value):
    global depth
    if code not in ours:
        return
    depth -= 1
    print("  " * depth + "<- " + code.co_name + " returned " + repr(value))


def unwound(code, offset, exception):
    global depth
    if code not in ours:
        return
    depth -= 1
    print("  " * depth + "<- " + code.co_name + " left with " + type(exception).__name__)


watching = event.PY_START | event.PY_RETURN | event.PY_UNWIND
monitoring.use_tool_id(3, "call tree")
try:
    monitoring.register_callback(3, event.PY_START, started)
    monitoring.register_callback(3, event.PY_RETURN, returned)
    monitoring.register_callback(3, event.PY_UNWIND, unwound)
    monitoring.set_events(3, watching)
    top()
finally:
    monitoring.set_events(3, 0)
    monitoring.free_tool_id(3)

Three `leaf` frames go on, the innermost one raises, and all three come off through `PY_UNWIND` rather than `PY_RETURN`. Then `top` catches it and returns normally. That is the frame stack unwinding, one frame per line, as it happens.

That cell uses `set_events`, which turns the events on for the whole process, and the callbacks throw away anything that is not one of our two functions. The cheaper call is `set_local_events`, which turns events on for one code object and leaves everything else in the process paying nothing. It is used further down for exactly that reason.

Not every event can be local. An event has to happen at a known instruction for the interpreter to be able to instrument that one spot, and on 3.14 the exception events did not qualify. On 3.15 they do. The next cell asks your own build which is which rather than taking either version's word for it.

In [ ]:
import sys

monitoring = sys.monitoring
event = monitoring.events


def nothing():
    pass


local = []
monitoring.use_tool_id(3, "asking")
try:
    for name in sorted(name for name in dir(event) if name.isupper()):
        value = getattr(event, name)
        if value == 0:
            continue
        try:
            monitoring.set_local_events(3, nothing.__code__, value)
        except ValueError:
            continue
        local.append(name)
finally:
    monitoring.set_local_events(3, nothing.__code__, 0)
    monitoring.free_tool_id(3)

print(f"{len(local)} events can be turned on for a single code object:")
print("   ", ", ".join(local))

## One instruction at a time

`INSTRUCTION` is the event that fires for every single instruction, and it is what a stepper is built on. One thing it will not give you is described below.

![static heights and observed order joined into one listing](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/t07-the-machine-runs/diagrams/where-the-numbers-come-from.svg)

Nothing in the standard library can read the values sitting on the value stack. Not `sys.monitoring`, not `sys.settrace`, not the frame object. Those values live in the frame's memory and there is no Python level door to them.

What we can do is join two things. The order instructions ran in is a real observation from `sys.monitoring`. The stack height at each offset is what `pyxray.stack` computed in T06 by walking the code object. Look up the second by the first and you get the height at every step of a real run, which is genuinely useful as long as nobody pretends it was measured.

`pyxray.stepper` does exactly that and its docstring says so. Here it is on a loop.

In [ ]:
from pyxray import stepper


def total_of(items):
    total = 0
    for item in items:
        total = total + item
    return total


recording = stepper.run(total_of, [1, 2, 3])
print("returned:", recording.result)
print("deepest the stack got:", recording.deepest)
print("co_stacksize says:", total_of.__code__.co_stacksize)
print()
print(recording.table())

Read the offset column rather than the step column. It counts up, then drops back to the offset of the `FOR_ITER`, three times over. That is the loop, and the drop is the back edge T06 taught you to spot in a listing, here being taken.

The exact offsets depend on your version, because inline cache sizes change between releases, and the shape stays the same.

The bars on the right are the stack height after each instruction. It peaks inside the loop body, when `LOAD_FAST_BORROW_LOAD_FAST_BORROW` puts both `total` and `item` on top of what was already there, then comes back down. That peak is exactly `co_stacksize`, which is the number T06 spent half a lesson working out, and this run used all of it.

The last `FOR_ITER` is the one that finds the list empty. After it comes `POP_ITER` and the loop is over.

### The instruction that never shows up

Count rows against a disassembly of `total_of` and one instruction is missing. There is an `END_FOR` between the last `FOR_ITER` and the `POP_ITER`, and it does not appear in the table above at all.

That is deliberate and it is written into the instruction's declaration: [Python/bytecodes.c:393-400@v3.15.0rc1#END_FOR](https://github.com/python/cpython/blob/v3.15.0rc1/Python/bytecodes.c#L393-L400). The `no_save_ip` marker means this instruction does not update the recorded instruction pointer, so as far as instrumentation is concerned it never becomes the current instruction. The comment explains why: `POP_ITER` needs to see the `FOR_ITER` as the instruction before it.

It is the kind of thing that would cost you an afternoon if you hit it without warning, so `pyxray` has a test pinning it rather than leaving it as a surprise.

In [ ]:
import dis

compiled = {item.offset: item.opname for item in dis.get_instructions(total_of)}
executed = {moment.offset for moment in recording.moments}

print("compiled but never reported:")
for offset, opname in compiled.items():
    if offset not in executed:
        print(f"   {offset:>4}  {opname}")

## Which way did the branch go

`INSTRUCTION` is the heaviest event there is, and most of the time you want less. `JUMP`, `BRANCH_LEFT` and `BRANCH_RIGHT` report only the moments where control could have gone two ways, and they say which way it went.

In [ ]:
import sys

monitoring = sys.monitoring
event = monitoring.events


def total_of(items):
    total = 0
    for item in items:
        total = total + item
    return total


def note(name):
    def callback(code, offset, destination):
        print(f"{name:<13} at {offset:>3}  went to {destination}")

    return callback


watching = event.JUMP | event.BRANCH_LEFT | event.BRANCH_RIGHT
monitoring.use_tool_id(3, "branches")
try:
    for name in ["JUMP", "BRANCH_LEFT", "BRANCH_RIGHT"]:
        monitoring.register_callback(3, getattr(event, name), note(name))
    monitoring.set_local_events(3, total_of.__code__, watching)
    total_of([1, 2])
finally:
    monitoring.set_local_events(3, total_of.__code__, 0)
    monitoring.free_tool_id(3)

Five lines for a two item loop. Every `BRANCH_LEFT` and `BRANCH_RIGHT` is at the same offset, and that offset is the `FOR_ITER`. Left means there was another item, right means there was not. The `JUMP` is the back edge, taken once per item except the last.

This is what a coverage tool wants. It does not care about every instruction, it cares about which edges of the graph were taken, and there are five of those here against twenty seven instructions.

## Why this is cheap

One design decision makes `sys.monitoring` different from everything before it.

![returning None keeps firing, returning DISABLE stops at that location](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/t07-the-machine-runs/diagrams/turning-an-event-off.svg)

A callback can return `sys.monitoring.DISABLE`. That does not turn the event off everywhere, it turns it off at that one code location, permanently, until somebody calls `restart_events`. A loop that runs a million times fires the callback once per instruction in the body and then goes quiet.

The next cell counts the calls both ways on the same five pass loop.

In [ ]:
import sys

monitoring = sys.monitoring
event = monitoring.events


def five_times():
    total = 0
    for n in range(5):
        total = total + n
    return total


def count(disable):
    seen = []
    monitoring.use_tool_id(3, "counting")
    try:

        def callback(code, offset):
            seen.append(offset)
            return monitoring.DISABLE if disable else None

        monitoring.register_callback(3, event.INSTRUCTION, callback)
        monitoring.set_local_events(3, five_times.__code__, event.INSTRUCTION)
        five_times()
    finally:
        monitoring.set_local_events(3, five_times.__code__, 0)
        monitoring.free_tool_id(3)
    return seen


for label, disable in [("returning None", False), ("returning DISABLE", True)]:
    seen = count(disable)
    print(f"{label:<18} {len(seen):>3} calls, {len(set(seen)):>3} distinct offsets")

Forty calls against fifteen: the loop body ran five times and the callback saw it once.

The old way is `sys.settrace`, which is what `pdb` and the original `coverage` are built on. It has one hook for the whole process, it fires on every line of every function once it is on, and there is no way to say "stop telling me about this one". Turning it on also switches the interpreter into a slower dispatch mode for everything, because instrumentation has to be checked between instructions: [Python/ceval_macros.h:128-141@v3.15.0rc1#DISPATCH_GOTO](https://github.com/python/cpython/blob/v3.15.0rc1/Python/ceval_macros.h#L128-L141) is where the tracing and non tracing tables diverge.

For comparison, here is `settrace` on a three pass loop.

In [ ]:
import sys
from collections import Counter


def three_times():
    total = 0
    for n in range(3):
        total = total + n
    return total


seen = Counter()


def trace(frame, kind, argument):
    if frame.f_code is three_times.__code__:
        frame.f_trace_opcodes = True
        seen[kind] += 1
        return trace
    return None


sys.settrace(trace)
three_times()
sys.settrace(None)

print(sum(seen.values()), "callbacks for a three pass loop")
for kind, number in seen.most_common():
    print(f"   {kind:<8} {number}")

Thirty nine calls, with no way to reduce them except by turning the whole thing off. `sys.monitoring` was added because debuggers and coverage tools were paying that price on every line of every program they touched.

`settrace` still works and is not going anywhere. For anything new, reach for the newer one.

## Frames from Python

The frame the interpreter uses is not a Python object, it is the block of memory from the diagram earlier. `PyFrameObject`, the thing you get from `sys._getframe()`, is built on demand and cached in the `frame_obj` field of that block, which you can see in the struct listing above.

The caching is visible from Python.

In [ ]:
import sys


def make_one():
    first = sys._getframe()
    second = sys._getframe()
    return first is second, first


same, escaped = make_one()
print("asked twice, got the same object:", same)
print("and it is still here after the call returned:", escaped)
print("its name:", escaped.f_code.co_name)

The frame object outliving the call is the whole reason frames are not on the C stack. A traceback holds onto frames, a generator is a frame that got paused, and a closure can keep one alive indefinitely. None of that would work if the frame went away when the C function returned.

Locals are worth one more cell, because there are two things that look the same and are not.

In [ ]:
import sys


def show():
    x = 1
    proxy = sys._getframe().f_locals
    snapshot = locals()
    print("f_locals is a", type(proxy).__name__)
    print("locals() is a", type(snapshot).__name__)

    proxy["x"] = 99
    print("after writing through f_locals, x is", x)

    snapshot["x"] = 1000
    print("after writing to the locals() dict, x is", x)


show()

`f_locals` is a live view onto the frame's slots, so writing through it changes the actual local. `locals()` inside a function is a plain dictionary copied out of those slots, so writing to it changes nothing. This used to be much more confusing than it is now, and the current behaviour was pinned down deliberately.

## The frame chain

Every frame has a pointer to the one that called it, which is `previous` in the struct. Walking that chain is what a traceback is.

In [ ]:
from pyxray import stepper


def third():
    for name, line in stepper.chain():
        print(f"{name:<20} line {line}")


def second():
    third()


def first():
    second()


first()

Innermost first, out to whatever is running the notebook. `stepper.chain` is nine lines and does nothing clever: take `sys._getframe()` and read `f_back` until it is `None`.

## Try it yourself

**One.** Run the stepper on a function with a `try` and an `except` in it, and raise something. Watch where the offsets jump to when the exception fires, then compare that with what `dis` shows for the exception table.

**Two.** Take the branch counting cell and turn it into a small coverage tool: record every `(offset, destination)` pair once, return `DISABLE`, and afterwards report which branches were never taken.

**Three.** `stepper.run` records the function you pass it and nothing it calls. Change it so it records a whole call tree by setting local events on the callee too when `PY_START` fires, then find out how much slower the recording is.

**Four.** Find the recursion depth your machine allows through `sorted`, then try again with the thread's stack size raised using the `threading.stack_size` function. The number should move.

## What just happened

The interpreter is one loop: read two bytes, look up the handler, run it, move the pointer on. Everything Python does is a handler inside that loop.

How the loop reaches the handler depends on the build: a `switch`, a computed goto through a jump table, or a tail call through a table of functions. Same results either way.

A frame is one block of memory holding the specials, the locals, and `co_stacksize` slots of working space. Frames go on a per thread stack, not on the C stack, so they can outlive the call.

Python calling Python pushes a frame and jumps rather than recursing in C, which is why ninety thousand deep is fine. Going out through a C function and back in grows the real C stack, which is a few megabytes and runs out at a few thousand.

`sys.monitoring` reports what the interpreter is doing, per code object, with a `DISABLE` return value that switches an event off at one location. That makes it cheap in a way `sys.settrace` never was.

Nothing in the standard library reads the values on the value stack. Joining observed instruction order with statically computed heights gets you most of the way, and it is worth being clear about which half is which.

## Where this goes next

You have now followed one line of Python from text all the way to a running instruction, which was the whole point of the first part.

T08 turns around and looks at what all those instructions have been pushing and popping. Every one of those values is a `PyObject`, every `PyObject` has a [type](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#type-object), and the type is where the behaviour lives. That is the start of the second half.